---
Manual Coding Behavior tool
---


Import
--

In [3]:
import os
import cv2
import pandas as pd
from pathlib import Path
import numpy as np

---
CONFIGURATION
---

In [4]:

# ------------ CONFIGURE HERE ------------ 
CSV_FILE = r"D:\\MasterThesis\\Manual_Annotation\\Sample\\Test_1\\filtered_data_chimps.csv" #← SINGLE CSV

VIDEO_FOLDER_CAM1 = r"D:\MasterThesis\Manual_Annotation\Sample\Test_1\videos\cam1"      # folder with MP4
VIDEO_FOLDER_CAM2 = r"D:\MasterThesis\Manual_Annotation\Sample\Test_1\videos\cam2"      # folder with MP4


OUTPUT_FOLDER_BASE = r"D:\MasterThesis\Manual_Annotation\Sample\Test_1\all_behaviors_writing"  

# Two subfolders, one per camera
OUTPUT_FOLDER_CAM1 = os.path.join(OUTPUT_FOLDER_BASE, "cam1")
OUTPUT_FOLDER_CAM2 = os.path.join(OUTPUT_FOLDER_BASE, "cam2")

OUTPUT_FOLDER_CAM1_DOUBT = os.path.join(OUTPUT_FOLDER_BASE, "cam1_team_review")
OUTPUT_FOLDER_CAM2_DOUBT = os.path.join(OUTPUT_FOLDER_BASE, "cam2_team_review")

os.makedirs(OUTPUT_FOLDER_CAM1_DOUBT, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_CAM2_DOUBT, exist_ok=True)

TRACKING_CSV = os.path.join(OUTPUT_FOLDER_BASE, "behavior_tracking_all.csv")

# ============== COLUMN NAMES ==============

COL_VIDEO = "observationid"                # column containing video filename
COL_MODIFIER = "modifier1"        # column with behavior labels
COL_MODIFIER2 = "modifier2"
COL_IMG_START = "imageindexstart"  # starting frame index for poking
COL_BEHAVIOR = "behavior"  #Full behavior description

# ============== CREATE OUTPUT FOLDERS ==============
os.makedirs(OUTPUT_FOLDER_BASE, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_CAM1, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_CAM2, exist_ok=True)

# ============== GLOBAL SYNC OFFSETS ==============
# Key: (csv_filename, video_name, behavior_name)  ← ADD behavior to key!

sync_offsets = {}

# ---------------------------------------


Helpers
-

In [5]:
def get_behavior_start_frame(row, fps: float) -> int:
    """
    Return the frame index to use as original_start for a behavior row.
    Priority:
      1) imageindexstart if available and not NaN
      2) starts (seconds) * fps, rounded to nearest int
    """
    # 1) Try imageindexstart
    img_start = row.get('imageindexstart', None)
    if img_start is not None and not pd.isna(img_start):
        try:
            return int(img_start)
        except (TypeError, ValueError):
            pass

    # 2) Fallback: use 'starts' in seconds
    starts_sec = row.get('starts', None)
    if starts_sec is not None and not pd.isna(starts_sec) and fps > 0:
        try:
            sec = float(starts_sec)
            frame = int(round(sec * fps))
            return max(0, frame)
        except (TypeError, ValueError):
            pass

    # If everything fails, return 0 as safe fallback
    return 0

In [6]:
def load_existing_tracking(tracking_csv_path: str) -> list[dict]:
    """
    Load existing tracking CSV if present.
    Returns a list of dicts. If file doesn't exist, returns [].
    """
    if os.path.isfile(tracking_csv_path):
        try:
            df_old = pd.read_csv(tracking_csv_path)
            print(f"📂 Loaded existing tracking: {tracking_csv_path} "
                  f"({len(df_old)} rows)")
            return df_old.to_dict(orient='records')
        except Exception as e:
            print(f"⚠️ Could not read existing tracking CSV: {e}")
            print("   Starting with an empty tracking list.")
            return []
    else:
        print(f"📄 No existing tracking file, starting fresh: {tracking_csv_path}")
        return []


def save_tracking_records(tracking_csv_path: str, tracking_records: list[dict]) -> None:
    """
    Save tracking_records to CSV, overwriting the file.
    Always use this together with load_existing_tracking(): we append in memory,
    then write out the full table.
    """
    if not tracking_records:
        print("⚠️ No tracking records to save.")
        return

    df_track = pd.DataFrame(tracking_records)
    df_track.to_csv(tracking_csv_path, index=False)
    print(f"📊 Tracking saved: {len(tracking_records)} rows → {tracking_csv_path}")

In [7]:
def save_tracking_records(tracking_csv_path: str, tracking_records: list[dict]) -> None:
    """
    Save tracking_records to CSV, overwriting the file.
    Always use this together with load_existing_tracking(): we append in memory,
    then write out the full table.
    """
    if not tracking_records:
        print("⚠️ No tracking records to save.")
        return

    df_track = pd.DataFrame(tracking_records)
    df_track.to_csv(tracking_csv_path, index=False)
    print(f"📊 Tracking saved: {len(tracking_records)} rows → {tracking_csv_path}")

In [8]:
def ask_recoded_behavior(original_behavior: str,
                         original_full_modifier: str) -> str:
    """
    Prompt the user for an alternative behavior label when they have doubt
    or disagree with the original coding.

    Returns the user-proposed behavior label (string).
    """
    print("\n📝 RECODING MODE")
    print(f"Original behavior (current token): {original_behavior}")
    print(f"Full Modifier #1: {original_full_modifier}")
    print("Type the behavior label you think is more appropriate.")
    print("Press Enter to keep the original label.")

    new_label = input("Your behavior label: ").strip()

    if new_label == "":
        print("↩️  No change, keeping original behavior label.")
        return original_behavior

    print(f"✅ New behavior label recorded: '{new_label}'")
    return new_label

In [9]:
def handle_doubt_clip(video_base: str,
                      behavior_name: str,
                      behavior_description: str,
                      behavior_context: str,
                      modifier1_full: str,
                      bout_label: str,
                      bout_start_frame,
                      bout_stop_frame,
                      original_start: int,
                      start_frame: int,
                      end_frame: int,
                      offset_cam1: int,
                      offset_cam2: int,
                      fps: float,
                      cam1_path: str,
                      cam2_path: str,
                      tracking_records: list,
                      csv_file_name: str) -> None:
    """
    Handle the 'doubt/disagree' decision:
    - Ask for recoded behavior label.
    - Ask which camera is useful.
    - Cut and save clips into *_DOUBT subfolders.
    - Append a tracking record with original + recoded behavior.
    """

    duration_frames = end_frame - start_frame + 1

    # 1) Ask user for their own label
    recoded_behavior = ask_recoded_behavior(
        original_behavior=behavior_name,
        original_full_modifier=modifier1_full
    )

    # 2) Build filenames
    safe_behavior = behavior_name.replace(' ', '_').replace('/', '-')
    safe_recoded = recoded_behavior.replace(' ', '_').replace('/', '-')

    out_name_cam1 = (f"{video_base}_cam1_{safe_behavior}_DOUBT_{safe_recoded}_"
                     f"{start_frame:06d}_{end_frame:06d}_{duration_frames:03d}f.mp4")
    out_name_cam2 = (f"{video_base}_cam2_{safe_behavior}_DOUBT_{safe_recoded}_"
                     f"{start_frame:06d}_{end_frame:06d}_{duration_frames:03d}f.mp4")

    out_path_cam1 = os.path.join(OUTPUT_FOLDER_CAM1_DOUBT, out_name_cam1)
    out_path_cam2 = os.path.join(OUTPUT_FOLDER_CAM2_DOUBT, out_name_cam2)

    # 3) Which camera is useful?
    print("Which cam is useful for team discussion? [1] cam1 | [2] cam2 | [Enter] both")
    cam_choice = input("Your choice: ").strip()

    if cam_choice == "1":
        useful_cam = "cam1"
    elif cam_choice == "2":
        useful_cam = "cam2"
    else:
        useful_cam = "cam1,cam2"

    # 4) Compute cut ranges with offsets
    cam1_start_cut = max(0, start_frame + offset_cam1)
    cam1_end_cut   = max(0, end_frame   + offset_cam1)
    cam2_start_cut = max(0, start_frame + offset_cam2)
    cam2_end_cut   = max(0, end_frame   + offset_cam2)

    # 5) Create tracking record
    record = {
        'video_name': video_base,
        'csv_file': csv_file_name,
        'status': 'doubt',
        'behavior_original': behavior_name,
        'behavior_recoded': recoded_behavior,
        'behavior_description': behavior_description,
        'behavior_context': behavior_context,
        'modifier1_full': modifier1_full,
        'bout_label': bout_label,
        'bout_start_frame': bout_start_frame,
        'bout_stop_frame': bout_stop_frame,
        'original_img_start': original_start,
        'final_img_start': start_frame,
        'final_img_stop': end_frame,
        'start_frame_diff': start_frame - original_start,
        'end_frame_diff': end_frame - original_start,
        'behavior_frames': duration_frames,
        'clip_filename_cam1': out_name_cam1,
        'clip_filename_cam2': out_name_cam2,
        'useful_cam': useful_cam,
        'offset_cam1': offset_cam1,
        'offset_cam2': offset_cam2,
        'offset_diff_cam2_minus_cam1': offset_cam2 - offset_cam1,
        'cam1_start_cut': cam1_start_cut,
        'cam1_end_cut': cam1_end_cut,
        'cam2_start_cut': cam2_start_cut,
        'cam2_end_cut': cam2_end_cut,
    }
    tracking_records.append(record)

    # 6) Cut and save clips
    print(f"💾 Saving DOUBT clips for '{behavior_name}' (recoded as '{recoded_behavior}')...")
    cut_and_save_segment(cam1_path, cam1_start_cut, cam1_end_cut, fps, out_path_cam1)
    cut_and_save_segment(cam2_path, cam2_start_cut, cam2_end_cut, fps, out_path_cam2)

    # Note: writing the CSV itself can stay in main()

Functions
-

In [10]:
def find_dual_video_paths(video_base: str) -> tuple[str, str]:
    """
    Find matching videos in BOTH cam1 and cam2 folders.
    NEW: Automatically tries .MP4 and .mp4 extensions.
    Returns (cam1_path, cam2_path) or raises error if any missing.
    """
    # Try both .MP4 and .mp4 extensions
    for ext in [".MP4", ".mp4"]:
        video_name = video_base + ext
        lower_target = video_name.lower()
        
        # ================= CAM1 =================
        cam1_candidate = os.path.join(VIDEO_FOLDER_CAM1, video_name)
        if not os.path.isfile(cam1_candidate):
            # Case-insensitive fallback
            cam1_candidate = None
            for f in os.listdir(VIDEO_FOLDER_CAM1):
                if f.lower() == lower_target:
                    cam1_candidate = os.path.join(VIDEO_FOLDER_CAM1, f)
                    break
        
        # ================= CAM2 =================  
        cam2_candidate = os.path.join(VIDEO_FOLDER_CAM2, video_name)
        if not os.path.isfile(cam2_candidate):
            # Case-insensitive fallback
            cam2_candidate = None
            for f in os.listdir(VIDEO_FOLDER_CAM2):
                if f.lower() == lower_target:
                    cam2_candidate = os.path.join(VIDEO_FOLDER_CAM2, f)
                    break
        
        # If BOTH found, return immediately
        if cam1_candidate and cam2_candidate:
            print(f"  ✅ DUAL VIDEO: cam1/{os.path.basename(cam1_candidate)} + "
                  f"cam2/{os.path.basename(cam2_candidate)}")
            return cam1_candidate, cam2_candidate
    
    # ================= NOT FOUND =================
    raise FileNotFoundError(
        f"Video not found in cam1 or cam2 folders: {video_base}.MP4 or .mp4"
    )

In [11]:
def get_dual_video_props(cam1_path: str, cam2_path: str) -> tuple:
    """
    NEW: Load properties for BOTH cameras, validate FPS match.
    Returns (cap1, cap2, fps, total_frames) - assumes synchronized videos.
    """
    cap1 = cv2.VideoCapture(cam1_path)
    cap2 = cv2.VideoCapture(cam2_path)
    
    if not cap1.isOpened() or not cap2.isOpened():
        raise RuntimeError(f"Cannot open dual video: cam1={cam1_path}, cam2={cam2_path}")
    
    fps1 = cap1.get(cv2.CAP_PROP_FPS)
    fps2 = cap2.get(cv2.CAP_PROP_FPS)
    total1 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
    total2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # ⚠️  CRITICAL: Validate synchronization
    if abs(fps1 - fps2) > 0.1:  # Allow 0.1 fps tolerance
        print(f"⚠️  FPS mismatch: cam1={fps1:.1f}, cam2={fps2:.1f}")
    if abs(total1 - total2) > 10:  # Allow small frame count diff
        print(f"⚠️  Frame count mismatch: cam1={total1}, cam2={total2}")
    
    fps = (fps1 + fps2) / 2  # Average FPS for playback
    total_frames = min(total1, total2)  # Use shorter duration
    
    return (cap1, cap2), fps, total_frames

In [12]:
def draw_modifier1_highlight_line(
    info_img,
    modifier1_full: str,
    current_behavior: str,
    x_start: int = 20,
    y: int = 35,
    font=cv2.FONT_HERSHEY_SIMPLEX,
    base_scale: float = 0.8
):
    """
    Draw the full 'Modifier #1' string on info_img, with the current behavior
    highlighted.

    - info_img: np.ndarray image to draw on.
    - modifier1_full: original 'Modifier #1' cell string
      (e.g. 'touching,pulling close,grabbing').
    - current_behavior: the behavior that is currently being coded
      (e.g. 'pulling close').
    - x_start, y: starting coordinates for the line.
    - font, base_scale: OpenCV text parameters.
    """
    # Split full modifier1 into tokens
    tokens = [t.strip() for t in modifier1_full.split(',') if t.strip()]

    x = x_start

    # Draw label 'MOD1:' in white
    label = "MOD1:"
    (label_w, _), _ = cv2.getTextSize(label, font, base_scale, 2)
    cv2.putText(info_img, label, (x, y),
                font, base_scale, (255, 255, 255), 2)
    x += label_w + 10

    # Draw tokens, highlighting the current behavior
    for i, tok in enumerate(tokens):
        # Comma separator (except before first token)
        if i > 0:
            sep = ","
            (sep_w, _), _ = cv2.getTextSize(sep, font, base_scale, 2)
            cv2.putText(info_img, sep, (x, y),
                        font, base_scale, (200, 200, 200), 2)
            x += sep_w + 8

        # Highlight if this token is the current behavior
        if tok == current_behavior:
            color = (0, 255, 255)   # yellow
            thickness = 3
        else:
            color = (180, 180, 180) # grey
            thickness = 2

        (tok_w, _), _ = cv2.getTextSize(tok, font, base_scale, thickness)
        cv2.putText(info_img, tok, (x, y),
                    font, base_scale, color, thickness)
        x += tok_w + 8

In [13]:
def play_dual_segment(caps: tuple,
                      start_frame: int,
                      end_frame: int,
                      fps: float,
                      behavior_name: str,
                      behavior_description: str,  # ← NEW: Full description from "Behavior" column
                      behavior_context: str,
                      modifier1_full: str,    
                      bout_label: str,
                      bout_start_frame: int | None,
                      bout_stop_frame: int | None,
                      offset_cam1: int = 0,
                      offset_cam2: int = 0):
    """
    Synchronized playback with BEHAVIOR NAME + DESCRIPTION displayed in INFO window.
    Returns (key_char, new_offset_cam1, new_offset_cam2).
    """
    cap1, cap2 = caps
    print(f"\n🎬 DUAL PLAY [{behavior_name}]: {behavior_description}")
    print(f"   Frames {start_frame}→{end_frame} ({end_frame-start_frame+1} frames)")
    print("V=Validate | M=Manual | B=Start+/- | E=End+/- | "
          "R=Restart | H=Hide | U=Sync | D=Doubt | N=Jump | ESC=Exit | Q=Quit")
    if bout_start_frame is not None and bout_stop_frame is not None:
        print(f"   Bout window: {bout_start_frame} → {bout_stop_frame} ({bout_label})")

    delay = int(1000 / fps) if fps > 0 else 33

    while True:
        # ============== Apply offsets and FORCE read to position ==============
        cap1.set(cv2.CAP_PROP_POS_FRAMES, max(0, start_frame + offset_cam1))
        cap2.set(cv2.CAP_PROP_POS_FRAMES, max(0, start_frame + offset_cam2))
        
        # Dummy read to ensure positioning works
        cap1.read()
        cap2.read()
        
        # Re-set position
        cap1.set(cv2.CAP_PROP_POS_FRAMES, max(0, start_frame + offset_cam1))
        cap2.set(cv2.CAP_PROP_POS_FRAMES, max(0, start_frame + offset_cam2))

        segment_done = False
        while True:
            current1 = int(cap1.get(cv2.CAP_PROP_POS_FRAMES))
            logical_current = current1 - offset_cam1

            if logical_current > end_frame:
                segment_done = True
                break

            ret1, frame1 = cap1.read()
            ret2, frame2 = cap2.read()
            if not ret1 or not ret2:
                segment_done = True
                break
            

            # ===== INFO WINDOW WITH HIGHLIGHTED MODIFIER1 =====
            info_h, info_w = 250, 1200
            info_img = np.zeros((info_h, info_w, 3), dtype=np.uint8)

            # Line 1: full Modifier #1 with current behavior highlighted
            draw_modifier1_highlight_line(
                info_img=info_img,
                modifier1_full=modifier1_full,
                current_behavior=behavior_name,
                x_start=20,
                y=35
            )

            # Other lines: description, context, bout, window
            y_desc = 75
            y_context = 115
            y_bout = 155
            y_window = 200

            txt2 = f"DESC: {behavior_description}"
            cv2.putText(info_img, txt2, (20, y_desc),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 0), 2)

            txt3 = f"MOD2 (context): {behavior_context}"
            cv2.putText(info_img, txt3, (20, y_context),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)

            if bout_start_frame is not None and bout_stop_frame is not None:
                txt4 = f"BOUT: {bout_label}  |  {bout_start_frame}-{bout_stop_frame}"
            else:
                txt4 = "BOUT: none"

            cv2.putText(info_img, txt4, (20, y_bout),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)

            txt5 = (f"WINDOW: {start_frame}-{end_frame}  |  "
                    f"frame {logical_current}  |  V/M/B/E/R/H/U/D/N/Q/ESC")
            cv2.putText(info_img, txt5, (20, y_window),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            cv2.imshow(WIN_CAM1, frame1)
            cv2.imshow(WIN_CAM2, frame2)
            cv2.imshow(WIN_INFO, info_img)

            key = cv2.waitKey(delay) & 0xFF
            if key == 27:
                return "esc", offset_cam1, offset_cam2
            
            if key in [ord('v'), ord('V'), ord('m'), ord('M'),
                       ord('b'), ord('B'), ord('e'), ord('E'),
                       ord('r'), ord('R'), ord('h'), ord('H'),
                       ord('q'), ord('Q'), ord('u'), ord('U'),
                       ord('d'), ord('D'), ord('n'), ord('N')]:
                key_char = chr(key).lower()
                return key_char, offset_cam1, offset_cam2

        # End of segment
        if segment_done:
            print(f"\n⏹️  End reached [{behavior_name}: {behavior_description}]. "
                  f"Choose V/M/B/E/R/H/U/Q.")
            while True:
                key = cv2.waitKey(0) & 0xFF

                if key == 27:  
                    return "esc", offset_cam1, offset_cam2
                
                if key in [ord('v'), ord('V'), ord('m'), ord('M'),
                           ord('b'), ord('B'), ord('e'), ord('E'),
                           ord('r'), ord('R'), ord('h'), ord('H'),
                           ord('q'), ord('Q'), ord('u'), ord('U'),
                           ord('d'), ord('D'), ord('n'), ord('N')]:
                    key_char = chr(key).lower()
                    return key_char, offset_cam1, offset_cam2

In [14]:
def cut_and_save_segment(input_path: str, start_frame: int, end_frame: int,
                         fps: float, output_path: str):
    """
    Cut [start_frame, end_frame] from input_path and save to output_path.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video for cutting: {input_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frame_count = 0
    while True:
        current = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        if current > end_frame:
            break

        ret, frame = cap.read()
        if not ret:
            break


        out.write(frame)
        frame_count += 1 

    cap.release()
    out.release()
    print(f"📹 Extracted {frame_count} clean frames to {output_path}")


In [15]:
def attach_bout_info(df: pd.DataFrame,
                     col_video: str,
                     col_behavior: str,
                     col_modifier: str,
                     col_img_start: str,
                     col_bout_start: str,
                     col_bout_stop: str) -> pd.DataFrame:
    """
    Attach bout information (from 'Manipulation time' rows) to every row.

    Rules:
    - A bout row is defined as:
        Behavior == 'Manipulation time' AND Modifier empty/NaN.
    - Its bout window is given by (col_bout_start, col_bout_stop).
    - All rows from the SAME video whose col_img_start/col_img_stop
      are inside this window inherit:
        _bout_start_frame, _bout_stop_frame, _bout_label.

    Returns a copy of df with new columns:
        _bout_start_frame, _bout_stop_frame, _bout_label, _is_bout_row
    """
    df = df.copy()

    # Initialize new columns
    df['_bout_start_frame'] = None
    df['_bout_stop_frame'] = None
    df['_bout_label'] = None
    df['_is_bout_row'] = False

    # Work per video, to avoid crossing individuals/sessions
    grouped = df.groupby(col_video, sort=False)

    updated_parts = []

    for video_name, df_video in grouped:
        # Work on a copy per video
        df_video = df_video.copy()

        # Ensure sorted by frame index
        df_video = df_video.sort_values(by=col_img_start).reset_index(drop=True)

        # Identify bout rows in this video
        # Behavior exactly "Manipulation time", and modifier empty/NaN
        is_bout = (
            df_video[col_behavior].astype(str).str.strip().eq("Manipulation time") &
            (df_video[col_modifier].isna() |
             (df_video[col_modifier].astype(str).str.strip() == ""))
        )

        df_video['_is_bout_row'] = is_bout

        # For convenience, map start/stop columns to integers where present
        # (bout rows should have full windows, others often have start==stop)
        bout_start_vals = df_video[col_bout_start]
        bout_stop_vals = df_video[col_bout_stop]

        # Now we propagate each bout to rows inside its window
        # We assume bouts do not overlap; if they do, the last bout wins.
        current_bout_start = None
        current_bout_stop = None
        current_bout_label = None

        bout_starts = []
        bout_stops = []
        bout_labels = []

        for idx_row, r in df_video.iterrows():
            if is_bout.iloc[idx_row]:
                # This row defines a new bout
                try:
                    current_bout_start = int(r[col_bout_start])
                except (TypeError, ValueError):
                    current_bout_start = None
                try:
                    current_bout_stop = int(r[col_bout_stop])
                except (TypeError, ValueError):
                    current_bout_stop = None

                # Use the Behavior text as label (here "Manipulation time")
                current_bout_label = str(r[col_behavior]).strip()

            # For the current row: if we have a valid bout window and the row's
            # start is inside it, attach the bout info.
            row_start = r[col_img_start]
            row_stop = r[col_bout_stop] if col_bout_stop in df_video.columns else row_start

            try:
                row_start = int(row_start)
            except (TypeError, ValueError):
                row_start = None
            try:
                row_stop = int(row_stop)
            except (TypeError, ValueError):
                row_stop = row_start

            if (
                current_bout_start is not None and
                current_bout_stop is not None and
                row_start is not None
            ):
                # Check if this row is inside the current bout window
                if (row_start >= current_bout_start) and (row_start <= current_bout_stop):
                    bout_starts.append(current_bout_start)
                    bout_stops.append(current_bout_stop)
                    bout_labels.append(current_bout_label)
                else:
                    bout_starts.append(None)
                    bout_stops.append(None)
                    bout_labels.append(None)
            else:
                bout_starts.append(None)
                bout_stops.append(None)
                bout_labels.append(None)

        df_video['_bout_start_frame'] = bout_starts
        df_video['_bout_stop_frame'] = bout_stops
        df_video['_bout_label'] = bout_labels

        updated_parts.append(df_video)

    df_out = pd.concat(updated_parts, ignore_index=True)
    return df_out

Main
-

In [16]:
def main():
    # ============== CREATE PERSISTENT WINDOWS ==============
    cv2.namedWindow(WIN_CAM1, cv2.WINDOW_NORMAL)
    cv2.namedWindow(WIN_CAM2, cv2.WINDOW_NORMAL)
    cv2.namedWindow(WIN_INFO, cv2.WINDOW_NORMAL)

    # ============== LOAD SINGLE AGGREGATED CSV ==============
    print(f"📄 Loading aggregated CSV: {CSV_FILE}")
    df_full = pd.read_csv(CSV_FILE)
    df_full = df_full.dropna(subset=[COL_MODIFIER, COL_BEHAVIOR], how='all')

    df_full = attach_bout_info(
        df_full,
        col_video=COL_VIDEO,
        col_behavior=COL_BEHAVIOR,
        col_modifier=COL_MODIFIER,
        col_img_start=COL_IMG_START,
        col_bout_start="imageindexstart",
        col_bout_stop="imageindexstop"
    )
    
    df_full = df_full.dropna(subset=[COL_MODIFIER])
    df_full = df_full[df_full[COL_MODIFIER].str.strip() != ""]
    
    print(f"✅ Loaded {len(df_full)} rows with behaviors")

    # ============== SPLIT MULTI-BEHAVIOR ROWS ==============
    behavior_rows = []
    for idx, row in df_full.iterrows():
        modifier_raw = str(row[COL_MODIFIER]).strip()
        behaviors = [b.strip() for b in modifier_raw.split(',') if b.strip()]
        for behavior in behaviors:
            new_row = row.copy()
            new_row['_split_behavior'] = behavior
            new_row['_modifier1_full'] = modifier_raw
            behavior_rows.append(new_row)
    
    df_behaviors = pd.DataFrame(behavior_rows)
    total_behavior_count = len(df_behaviors)
    
    print(f"\n🎯 TOTAL: {total_behavior_count} behavior windows to validate")
    print(f"   (includes split behaviors from comma-separated values)")
    print(f"{'='*70}\n")

    if total_behavior_count == 0:
        print("❌ No behaviors found!")
        return

    # ============== TRACKING & PROCESSING ==============
    tracking_records = load_existing_tracking(TRACKING_CSV)
    processed_count = 0
    current_index = 0

    # Arrow key codes on your system (from waitKeyEx test)
    LEFT_ARROW = 37
    RIGHT_ARROW = 39

    # ============== ITERATE THROUGH ALL BEHAVIORS ==============
    while current_index < total_behavior_count:
        row = df_behaviors.iloc[current_index]
        processed_count = current_index + 1

        # Extract metadata
        video_base = row[COL_VIDEO]
        behavior_name = row['_split_behavior']

        original_modifier_full = str(row.get('_modifier1_full', behavior_name))
        behavior_description = str(row[COL_BEHAVIOR]).strip()
        behavior_context = str(row.get(COL_MODIFIER2, "")).strip()

        bout_start_frame = row.get('_bout_start_frame')
        bout_stop_frame = row.get('_bout_stop_frame')
        bout_label = row.get('_bout_label')
        
        if behavior_description == 'nan' or behavior_description == '':
            behavior_description = "(no description)"
        
        if pd.isna(bout_start_frame):
            bout_start_frame = None
        if pd.isna(bout_stop_frame):
            bout_stop_frame = None
        if pd.isna(bout_label) or str(bout_label).strip() == "":
            bout_label = "Manipulation time"
        else:
            bout_label = str(bout_label).strip()

        # ============== FIND VIDEOS ==============
        try:
            cam1_path, cam2_path = find_dual_video_paths(video_base)
        except FileNotFoundError as e:
            print(f"❌ {e}")
            record = {
                'video_name': video_base,
                'behavior': f'{behavior_name}_VIDEO_MISSING',
                'original_img_start': original_start,
                'final_img_start': -1,
                'final_img_stop': -1,
                'remarks': 'Video files not found'
            }
            tracking_records.append(record)
            save_tracking_records(TRACKING_CSV, tracking_records)
            current_index += 1
            continue

        # ============== LOAD VIDEO PROPERTIES ==============
        caps, fps, total_frames = get_dual_video_props(cam1_path, cam2_path)

        # ============== COMPUTE ORIGINAL START ROBUSTLY ====
        original_start = get_behavior_start_frame(row,fps)

        print(f"\n{'='*60}")
        print(f"[{processed_count}/{total_behavior_count}] "
              f"🎥 {video_base} | Behavior: '{behavior_name}' | Frame: {original_start}")
        print(f"{'='*60}")
        
        # ============== CREATE FRAME WINDOW ================
        start_frame = max(0, original_start - 25)
        end_frame = min(total_frames - 1, original_start + 35)

        # ============== RETRIEVE SYNC OFFSETS ==============
        key_sync = (os.path.basename(CSV_FILE), video_base, behavior_name)
        offset_cam1, offset_cam2 = sync_offsets.get(key_sync, (0, 0))

        remaining = total_behavior_count - processed_count + 1
        print(f"   ⏳ Remaining: {remaining} behaviors")
        print(f"   | Window: frames {start_frame} → {end_frame}")

        # ============== INTERACTIVE VALIDATION LOOP ==============
        while True:
            print(f"\n🔄 [{behavior_name}] {behavior_description}")
            print(f"   Frames {start_frame} → {end_frame}")
            print("V=Validate | M=Manual | B=Start+/- | E=End+/- | "
                  "R=Replay | H=Hide | U=Sync | D=TeamReview | N=Jump | Q=Quit | ESC=Exit")

            key, offset_cam1, offset_cam2 = play_dual_segment(
                caps, start_frame, end_frame, fps,
                behavior_name=behavior_name,
                behavior_description=behavior_description,
                behavior_context=behavior_context,
                modifier1_full=original_modifier_full,
                bout_label=bout_label,
                bout_start_frame=bout_start_frame,
                bout_stop_frame=bout_stop_frame,
                offset_cam1=offset_cam1,
                offset_cam2=offset_cam2
            )
            
            sync_offsets[key_sync] = (offset_cam1, offset_cam2)
            
            print(f"→ User pressed: '{key}' "
                  f"(offsets: cam1={offset_cam1}, cam2={offset_cam2})")

            # ESC: exit script
            if key == "esc":
                print("🧨 ESC pressed → exiting script.")
                for c in caps:
                    if c.isOpened():
                        c.release()
                cv2.destroyAllWindows()
                save_tracking_records(TRACKING_CSV, tracking_records)
                return

            # U: sync offsets
            if key == 'u':
                offset_cam1, offset_cam2 = adjust_sync_offsets(offset_cam1, offset_cam2)
                sync_offsets[key_sync] = (offset_cam1, offset_cam2)
                continue
            
            # N: navigation with arrows + numeric index
            elif key == 'n':
                print("\n🧭 NAVIGATION MODE")
                print("Use:")
                print("  → (right arrow)      : go to next behavior")
                print("  ← (left arrow)       : go to previous behavior")
                print("  Type number + Enter  : jump to behavior #")
                print("  V or Enter           : confirm current behavior and exit nav")
                print("  Q                    : cancel navigation (stay here)")
                print("  ESC                  : exit script")
                print(f"Current: behavior #{current_index + 1} / {total_behavior_count}")

                # Release current caps; will reopen around new index
                for c in caps:
                    if c.isOpened():
                        c.release()

                number_buffer = ""  # collect digits typed by the user

                while True:
                    key_nav = cv2.waitKeyEx(0)

                    # ESC in navigation
                    if key_nav == 27:
                        print("🧨 ESC pressed in navigation → exiting script.")
                        cv2.destroyAllWindows()
                        save_tracking_records(TRACKING_CSV, tracking_records)
                        return

                    # LEFT arrow
                    if key_nav == LEFT_ARROW:
                        if current_index > 0:
                            current_index -= 1
                            number_buffer = ""
                            print(f"⬅️  Moved to behavior #{current_index + 1}")
                        else:
                            print("🚫 Already at first behavior.")
                        continue

                    # RIGHT arrow
                    if key_nav == RIGHT_ARROW:
                        if current_index < total_behavior_count - 1:
                            current_index += 1
                            number_buffer = ""
                            print(f"➡️  Moved to behavior #{current_index + 1}")
                        else:
                            print("🚫 Already at last behavior.")
                        continue

                    # DIGITS: build number buffer
                    if ord('0') <= key_nav <= ord('9'):
                        digit = chr(key_nav)
                        number_buffer += digit
                        print(f"   [typing index] {number_buffer}")
                        continue

                    # Enter or V: if number_buffer has content, jump there; otherwise confirm
                    if key_nav in (ord('\r'), ord('\n'), ord('v'), ord('V')):
                        if number_buffer:
                            try:
                                target = int(number_buffer)
                                if 1 <= target <= total_behavior_count:
                                    current_index = target - 1
                                    print(f"✅ Jump to behavior #{current_index + 1}")
                                else:
                                    print(f"❌ {target} is out of range 1–{total_behavior_count}")
                            except ValueError:
                                print("❌ Invalid number.")
                        else:
                            print(f"✅ Navigation confirmed at behavior #{current_index + 1}")
                        break  # exit navigation mode

                    # Q: cancel navigation
                    if key_nav in (ord('q'), ord('Q')):
                        print("↩️  Navigation cancelled, staying on current behavior.")
                        break

                    # Other keys ignored
                    continue

                # Leave inner loop; outer while restarts at updated current_index
                break

            elif key == 'q':
                print(f"👋 QUIT at #{processed_count} | Remaining: {remaining}")
                for c in caps:
                    if c.isOpened():
                        c.release()
                break
            
            elif key == 'v':
                print(f"✅ VALIDATED #{processed_count} | Remaining: {remaining}")
                duration_frames = end_frame - start_frame + 1

                safe_behavior = behavior_name.replace(' ', '_').replace('/', '-')
                out_name_cam1 = (f"{video_base}_cam1_{safe_behavior}_"
                                 f"{start_frame:06d}_{end_frame:06d}_{duration_frames:03d}f.mp4")
                out_name_cam2 = (f"{video_base}_cam2_{safe_behavior}_"
                                 f"{start_frame:06d}_{end_frame:06d}_{duration_frames:03d}f.mp4")
                
                out_path_cam1 = os.path.join(OUTPUT_FOLDER_CAM1, out_name_cam1)
                out_path_cam2 = os.path.join(OUTPUT_FOLDER_CAM2, out_name_cam2)
                
                print("Which cam is useful? [1] cam1 | [2] cam2 | [Enter] both")
                cam_choice = input("Your choice: ").strip()
                
                if cam_choice == "1":
                    useful_cam = "cam1"
                elif cam_choice == "2":
                    useful_cam = "cam2"
                else:
                    useful_cam = "cam1,cam2"

                cam1_start_cut = max(0, start_frame + offset_cam1)
                cam1_end_cut = max(0, end_frame + offset_cam1)
                cam2_start_cut = max(0, start_frame + offset_cam2)
                cam2_end_cut = max(0, end_frame + offset_cam2)

                record = {
                    'video_name': video_base,
                    'csv_file': os.path.basename(CSV_FILE),
                    'behavior': behavior_name,
                    'behavior_description': behavior_description,
                    'behavior_context': behavior_context,
                    'bout_label': bout_label,
                    'bout_start_frame': bout_start_frame,
                    'bout_stop_frame': bout_stop_frame,
                    'original_img_start': original_start,
                    'final_img_start': start_frame,
                    'final_img_stop': end_frame,
                    'start_frame_diff': start_frame - original_start,
                    'end_frame_diff': end_frame - original_start,
                    'behavior_frames': duration_frames,
                    'clip_filename_cam1': out_name_cam1,
                    'clip_filename_cam2': out_name_cam2,
                    'useful_cam': useful_cam,
                    'offset_cam1': offset_cam1,
                    'offset_cam2': offset_cam2,
                    'offset_diff_cam2_minus_cam1': offset_cam2 - offset_cam1,
                    'cam1_start_cut': cam1_start_cut,
                    'cam1_end_cut': cam1_end_cut,
                    'cam2_start_cut': cam2_start_cut,
                    'cam2_end_cut': cam2_end_cut,
                }
                tracking_records.append(record)                       
                
                print(f"💾 Saving clips for '{behavior_name}'...")
                cut_and_save_segment(cam1_path, cam1_start_cut, cam1_end_cut, fps, out_path_cam1)
                cut_and_save_segment(cam2_path, cam2_start_cut, cam2_end_cut, fps, out_path_cam2)
                
                save_tracking_records(TRACKING_CSV, tracking_records)
                
                for c in caps:
                    if c.isOpened():
                        c.release()
                break

            elif key == 'h':
                remarks = input("Remarks (or Enter for 'bad_quality'): ").strip() or "bad_quality"
                print(f"⏭️  HIDDEN #{processed_count} | Remaining: {remaining}")

                record = {
                    'video_name': video_base,
                    'csv_file': os.path.basename(CSV_FILE),
                    'behavior': f'{behavior_name}_HIDDEN',
                    'behavior_description': behavior_description,
                    'bout_label': bout_label,
                    'bout_start_frame': bout_start_frame,
                    'bout_stop_frame': bout_stop_frame,
                    'original_img_start': original_start,
                    'final_img_start': -1,
                    'final_img_stop': -1,
                    'start_frame_diff': 0,
                    'end_frame_diff': 0,
                    'behavior_frames': 0,
                    'clip_filename_cam1': '',
                    'clip_filename_cam2': '',
                    'useful_cam': '',
                    'remarks': remarks
                }
                tracking_records.append(record)
                
                save_tracking_records(TRACKING_CSV, tracking_records)
                
                for c in caps:
                    if c.isOpened():
                        c.release()
                break

            elif key == 'r':
                print("🔄 Replaying...")
                continue

            elif key == 'm':
                print("\n📝 MANUAL mode")
                try:
                    new_start = int(input(f"New START (0–{total_frames-1}): "))
                    new_end = int(input(f"New END ({new_start}–{total_frames-1}): "))
                    start_frame = max(0, min(new_start, total_frames - 1))
                    end_frame = max(start_frame, min(new_end, total_frames - 1))
                    print(f"✅ Updated: {start_frame} → {end_frame}")
                except ValueError:
                    print("❌ Invalid input")
                continue

            elif key == 'b':
                print("\n⬅️  Adjust START")
                try:
                    offset = int(input("Offset (+/- frames): "))
                    start_frame = max(0, min(start_frame + offset, end_frame))
                    print(f"✅ New start: {start_frame}")
                except ValueError:
                    print("❌ Invalid offset")
                continue

            elif key == 'e':
                print("\n➡️  Adjust END")
                try:
                    offset = int(input("Offset (+/- frames): "))
                    end_frame = max(start_frame, min(end_frame + offset, total_frames - 1))
                    print(f"✅ New end: {end_frame}")
                except ValueError:
                    print("❌ Invalid offset")
                continue

            elif key == 'd':
                print(f"❓ TEAM REVIEW on #{processed_count} | Remaining: {remaining}")

                handle_doubt_clip(
                    video_base=video_base,
                    behavior_name=behavior_name,
                    behavior_description=behavior_description,
                    behavior_context=behavior_context,
                    modifier1_full=original_modifier_full,
                    bout_label=bout_label,
                    bout_start_frame=bout_start_frame,
                    bout_stop_frame=bout_stop_frame,
                    original_start=original_start,
                    start_frame=start_frame,
                    end_frame=end_frame,
                    offset_cam1=offset_cam1,
                    offset_cam2=offset_cam2,
                    fps=fps,
                    cam1_path=cam1_path,
                    cam2_path=cam2_path,
                    tracking_records=tracking_records,
                    csv_file_name=os.path.basename(CSV_FILE),
                )

                save_tracking_records(TRACKING_CSV, tracking_records)
                
                for c in caps:
                    if c.isOpened():
                        c.release()
                break

        # Advance to next behavior by default
        current_index += 1

    cv2.destroyAllWindows()

    validated = sum(1 for r in tracking_records
                    if 'behavior' in r and '_HIDDEN' not in r['behavior']
                    and '_MISSING' not in r['behavior'])
    hidden = sum(1 for r in tracking_records
                 if 'behavior' in r and '_HIDDEN' in r['behavior'])
    missing = sum(1 for r in tracking_records
                  if 'behavior' in r and '_MISSING' in r['behavior'])
    
    print(f"\n🎉 FINISHED!")
    print(f"   ✅ Validated: {validated} behaviors")
    print(f"   ⏭️  Hidden: {hidden} behaviors")
    print(f"   ❌ Missing videos: {missing} behaviors")
    print(f"   📊 Total tracked: {len(tracking_records)} | {TRACKING_CSV}")

---
Synchronisation function
--


In [17]:
def adjust_sync_offsets(offset_cam1: int, offset_cam2: int) -> tuple[int, int]:
    """
    Interactive synchronisation: choose cam (1/2) and +/- frames.
    Returns updated (offset_cam1, offset_cam2).

    This function is called when the user presses Y or U to align the two cameras.
    """
    print("\n🧭 SYNCHRONISATION MODE")
    print("Which camera do you want to shift?")
    print("1 = CAM1, 2 = CAM2, Enter = cancel")
    cam_choice = input("Cam to shift (1/2 or Enter): ").strip()

    if cam_choice not in ["1", "2"]:
        print("↩️  Sync cancelled.")
        return offset_cam1, offset_cam2

    try:
        delta = int(input("Offset (+/- frames, positive = later, negative = earlier): "))
    except ValueError:
        print("❌ Invalid number, sync cancelled.")
        return offset_cam1, offset_cam2

    if cam_choice == "1":
        offset_cam1 += delta
        print(f"✅ CAM1 offset is now {offset_cam1} frames.")
    else:
        offset_cam2 += delta
        print(f"✅ CAM2 offset is now {offset_cam2} frames.")

    return offset_cam1, offset_cam2

In [18]:
# Global window names (constants)
WIN_CAM1 = "Behavior validation - CAM1"
WIN_CAM2 = "Behavior validation - CAM2"
WIN_INFO = "Behavior validation - INFO"

RUN Here 
-

In [19]:
5
if __name__ == "__main__":
    main()
    

📄 Loading aggregated CSV: D:\\MasterThesis\\Manual_Annotation\\Sample\\Test_1\\filtered_data_chimps.csv
✅ Loaded 464 rows with behaviors

🎯 TOTAL: 564 behavior windows to validate
   (includes split behaviors from comma-separated values)

📄 No existing tracking file, starting fresh: D:\MasterThesis\Manual_Annotation\Sample\Test_1\all_behaviors_writing\behavior_tracking_all.csv
  ✅ DUAL VIDEO: cam1/Frederika_29.7.22_Test1_T1.MP4 + cam2/Frederika_29.7.22_Test1_T1.MP4
⚠️  Frame count mismatch: cam1=32016, cam2=32136

[1/564] 🎥 Frederika_29.7.22_Test1_T1 | Behavior: 'touching' | Frame: 702
   ⏳ Remaining: 564 behaviors
   | Window: frames 677 → 737

🔄 [touching] Touch: left hand
   Frames 677 → 737
V=Validate | M=Manual | B=Start+/- | E=End+/- | R=Replay | H=Hide | U=Sync | D=TeamReview | N=Jump | Q=Quit | ESC=Exit

🎬 DUAL PLAY [touching]: Touch: left hand
   Frames 677→737 (61 frames)
V=Validate | M=Manual | B=Start+/- | E=End+/- | R=Restart | H=Hide | U=Sync | D=Doubt | N=Jump | ESC=Exit